In [8]:
import open3d as o3d
import numpy as np
import json
import os
from pathlib import Path
from glob import glob
from PIL import Image
import cv2
import networkx as nx
import matplotlib.pyplot as plt
import random
from PIL import Image
import os
import images_processing
from PIL import Image, ImageDraw

import re
import trimesh
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import community as community_louvain


In [2]:
# scene_dir = 'ScanNet_Data/data/56a0ec536c'

# Change the scene: c9abde4c4b - toilet
# scene: 285efbc7cf - kitchen
scene_dir = 'ScanNet_Data/data/0a7cc12c0e'

# Change the scene: 0a7cc12c0e - bedroom
# scene_dir = 'ScanNet_Data/data/0a7cc12c0e'
mesh_file = scene_dir + '/scans/mesh_aligned_0.05_semantic.ply'
segments_file = scene_dir + '/scans/segments.json'
anno_file = scene_dir + '/scans/segments_anno.json'
output_dir = scene_dir + '/extracted_individual_objects'
os.makedirs(output_dir, exist_ok=True)
colmap_dir = os.path.join(scene_dir, 'dslr/colmap')
image_dir = os.path.join(scene_dir, 'dslr/resized_images')
region_crop_dir = scene_dir + '/region_crops'
os.makedirs(region_crop_dir, exist_ok=True)
region_dir = scene_dir + '/regions'
os.makedirs(region_dir, exist_ok=True)
region_top5_dir = os.path.join(scene_dir, 'region_cropped_top5')
os.makedirs(region_top5_dir, exist_ok=True)

In [3]:
mesh = o3d.io.read_triangle_mesh(mesh_file)
vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)
segment_ids = np.asarray(json.load(open(segments_file))['segIndices'])
anno_data = np.asarray(json.load(open(anno_file))['segGroups'])

In [4]:
segment_to_object = {}
object_to_label = {}
for object in anno_data:
    object_id = object['objectId']
    label = object['label']
    segments = object['segments']
    for segment_id in segments:
        segment_to_object[segment_id] = object_id
    object_to_label[object_id] = label

In [5]:
unique_instances = set(segment_to_object.values())
print(f'There are {len(unique_instances)} unique instances.')

There are 140 unique instances.


## Caption Generation Experiment

In [6]:
import internVL
vlm = internVL.InternVL_VLM()

Some parameters are on the meta device because they were offloaded to the cpu.


## Scene Graph Surface to Surface Distance

### Save the captions with: 

object_id, object_category, object_surrounding_caption, object_surrounding_caption_embedding, object_centroid

In [26]:
with open("object_captions_latest_version.json", "r", encoding="utf-8") as f:
    objects = json.load(f)    

In [9]:
import os
import re
import numpy as np
import trimesh
from scipy.spatial import cKDTree
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import community as community_louvain
import pandas as pd
import plotly.express as px

# ---------------------------------------------
# ⚡️ PARAMETERS
# ---------------------------------------------
SIM_THRESHOLD = 0.5
ALPHA = 0.5
BETA = 1

mesh_dir = os.path.join(scene_dir, 'extracted_individual_objects')       # <- 替换成你的路径
objects_json = "object_captions_latest_version.json"  # <- 替换成你的路径

# ---------------------------------------------
# ⚡️ 1. Load Objects Metadata (with embeddings etc.)
# ---------------------------------------------
import json
with open(objects_json) as f:
    objects = json.load(f)

# ---------------------------------------------
# ⚡️ 2. Load Mesh Vertices
# ---------------------------------------------
def get_object_number(fname):
    m = re.match(r'object_(\d+)_', fname)
    return int(m.group(1)) if m else float('inf')

mesh_files = [
    f for f in os.listdir(mesh_dir) 
    if f.endswith('.ply') and 'REMOVE' not in f
]
mesh_files_sorted = sorted(mesh_files, key=get_object_number)

mesh_points_list = []
mesh_ids = []
for fname in mesh_files_sorted:
    mesh_path = os.path.join(mesh_dir, fname)
    mesh = trimesh.load(mesh_path, process=False)
    vertices = np.array(mesh.vertices)
    mesh_points_list.append(vertices)
    # Try to match ID from filename
    obj_num = get_object_number(fname)
    mesh_ids.append(obj_num)

print(f"✅ Loaded {len(mesh_points_list)} meshes.")

# ---------------------------------------------
# ⚡️ 3. Build Mesh-to-Mesh Shortest Distance Matrix
# ---------------------------------------------
n = len(mesh_points_list)
mesh_dist_matrix = np.zeros((n, n))

for i in range(n):
    tree_i = cKDTree(mesh_points_list[i])
    for j in range(i + 1, n):
        tree_j = cKDTree(mesh_points_list[j])
        dist_ij, _ = tree_i.query(mesh_points_list[j])
        dist_ji, _ = tree_j.query(mesh_points_list[i])
        min_dist = min(dist_ij.min(), dist_ji.min())
        mesh_dist_matrix[i, j] = min_dist
        mesh_dist_matrix[j, i] = min_dist

print("✅ Computed mesh-to-mesh shortest distances.")

# Normalize distances
min_dist = np.min(mesh_dist_matrix)
max_dist = np.max(mesh_dist_matrix)
norm_mesh_dist_matrix = (mesh_dist_matrix - min_dist) / (max_dist - min_dist + 1e-8)
print("✅ Normalized distance matrix.")

# ---------------------------------------------
# ⚡️ 4. Compute Textual Cosine Similarity
# ---------------------------------------------
id_to_index = {obj['object_id']: i for i, obj in enumerate(objects)}

embeddings = []
ids = []
for obj in objects:
    if obj["object_id"] in mesh_ids:
        embeddings.append(np.array(obj["embeddings"]))
        ids.append(obj["object_id"])

embeddings = np.vstack(embeddings)
text_sim_matrix = cosine_similarity(embeddings)
print("✅ Computed cosine similarity matrix.")

# ---------------------------------------------
# ⚡️ 5. Build Scene Graph
# ---------------------------------------------
G = nx.Graph()
for obj in objects:
    if obj['object_id'] in mesh_ids:
        G.add_node(obj['object_id'], **obj)

edge_count = 0
for i in range(n):
    for j in range(i + 1, n):
        sim = text_sim_matrix[i, j]
        if sim < SIM_THRESHOLD:
            continue
        dist_norm = norm_mesh_dist_matrix[i, j]
        weight = ALPHA * sim - BETA * dist_norm
        if weight > 0:
            G.add_edge(
                mesh_ids[i], mesh_ids[j],
                weight=weight,
                similarity=sim,
                distance=mesh_dist_matrix[i, j]
            )
            edge_count += 1

print(f"✅ Graph built with {G.number_of_nodes()} nodes and {edge_count} edges (sim > {SIM_THRESHOLD}).")

# ---------------------------------------------
# ⚡️ 6. Community Detection
# ---------------------------------------------
partition = community_louvain.best_partition(G, weight='weight')
print(f"✅ Detected regions: {set(partition.values())}")

for obj in objects:
    if obj['object_id'] in mesh_ids:
        obj['detected_region'] = partition.get(obj['object_id'], -1)

print("✅ Assigned detected_region to objects.")

# ---------------------------------------------
# ⚡️ 7. 3D Visualization
# ---------------------------------------------
plot_data = []
for obj in objects:
    if obj['object_id'] in mesh_ids and 'centroid' in obj:
        centroid = obj['centroid']
        plot_data.append({
            "id": obj["object_id"],
            "x": centroid[0],
            "y": centroid[1],
            "z": centroid[2],
            "category": obj.get("object_category", "unknown"),
            "region": str(obj.get("detected_region", "unknown"))
        })

df = pd.DataFrame(plot_data)

fig = px.scatter_3d(
    df,
    x='x', y='y', z='z',
    color='region',
    hover_data=['id', 'region', 'category'],
    title="3D Visualization with Mesh Surface Distances",
    opacity=0.8
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(scene=dict(aspectmode='data'))
fig.show()

print("✅ 3D visualization complete.")

✅ Loaded 136 meshes.
✅ Computed mesh-to-mesh shortest distances.
✅ Normalized distance matrix.
✅ Computed cosine similarity matrix.
✅ Graph built with 136 nodes and 3355 edges (sim > 0.5).
✅ Detected regions: {0, 1, 2, 3}
✅ Assigned detected_region to objects.


✅ 3D visualization complete.


## Scene Graph

In [35]:
import numpy as np
import networkx as nx
import pandas as pd
import plotly.express as px
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist
import community as community_louvain

# ------------------------------------------------
# ⚡️ PARAMETERS
# ------------------------------------------------
SIM_THRESHOLD = 0.5   # 语义相似度阈值
ALPHA = 0.5         # 文本相似度的权重
BETA  = 1           # 空间距离惩罚的权重

# ------------------------------------------------
# ⚡️ 1. Prepare Data
# ------------------------------------------------
ids = []
embeddings = []
centroids = []

for obj in objects:
    if "embeddings" in obj and "centroid" in obj:
        ids.append(obj["object_id"])
        embeddings.append(np.array(obj["embeddings"]))
        centroids.append(np.array(obj["centroid"]))

embeddings = np.vstack(embeddings)
centroids = np.vstack(centroids)

print(f"✅ Loaded {len(ids)} objects with embeddings and centroids.")

# ------------------------------------------------
# ⚡️ 2. Compute Textual Cosine Similarity
# ------------------------------------------------
text_sim_matrix = cosine_similarity(embeddings)
print("✅ Computed cosine similarity matrix.")

# ------------------------------------------------
# ⚡️ 3. Compute Spatial Distance
# ------------------------------------------------
space_dist_matrix = cdist(centroids, centroids, metric='euclidean')
print("✅ Computed Euclidean distance matrix.")

# Normalize distances to [0, 1]
min_dist = np.min(space_dist_matrix)
max_dist = np.max(space_dist_matrix)
norm_space_dist_matrix = (space_dist_matrix - min_dist) / (max_dist - min_dist + 1e-8)
print("✅ Normalized distance matrix to [0,1].")

# ------------------------------------------------
# ⚡️ 4. Build Graph
# ------------------------------------------------
G = nx.Graph()

# Add all nodes with attributes
for obj in objects:
    G.add_node(obj['object_id'], **obj)

# Add edges only if similarity > threshold
n = len(ids)
edge_count = 0
for i in range(n):
    for j in range(i + 1, n):
        sim = text_sim_matrix[i, j]
        if sim < SIM_THRESHOLD:
            continue

        dist_norm = norm_space_dist_matrix[i, j]
        weight = ALPHA * sim - BETA * dist_norm

        # Keep only positive weight edges
        if weight > 0:
            G.add_edge(ids[i], ids[j], weight=weight, similarity=sim, distance=space_dist_matrix[i, j])
            edge_count += 1

print(f"✅ Graph built with {G.number_of_nodes()} nodes and {edge_count} edges (sim > {SIM_THRESHOLD}).")

# ------------------------------------------------
# ⚡️ 5. Community Detection
# ------------------------------------------------
partition = community_louvain.best_partition(G, weight='weight')
print(f"✅ Detected regions: {set(partition.values())}")

# Assign back to objects
for obj in objects:
    obj['detected_region'] = partition[obj['object_id']]

print("✅ Assigned detected_region to objects.")

# ------------------------------------------------
# ⚡️ 6. 3D Visualization
# ------------------------------------------------
plot_data = []
for obj in objects:
    centroid = obj.get("centroid")
    if centroid is not None:
        plot_data.append({
            "id": obj["object_id"],
            "x": centroid[0],
            "y": centroid[1],
            "z": centroid[2],
            "category": obj.get("object_category", "unknown"),
            "region": str(obj.get("detected_region", "unknown"))
        })

df = pd.DataFrame(plot_data)

fig = px.scatter_3d(
    df,
    x='x', y='y', z='z',
    color='region',
    hover_data=['id', 'region', 'category'],
    title=f"3D Visualization of Object Centroids with Detected Regions (Sim > {SIM_THRESHOLD})",
    opacity=0.8
)

fig.update_traces(marker=dict(size=5))
fig.update_layout(scene=dict(aspectmode='data'))
fig.show()

print("✅ 3D visualization complete.")

✅ Loaded 140 objects with embeddings and centroids.
✅ Computed cosine similarity matrix.
✅ Computed Euclidean distance matrix.
✅ Normalized distance matrix to [0,1].
✅ Graph built with 140 nodes and 2638 edges (sim > 0.5).
✅ Detected regions: {0, 1, 2, 3, 4}
✅ Assigned detected_region to objects.


✅ 3D visualization complete.


In [37]:
with open('object_captions_debug.json', 'w', encoding='utf-8') as f:
    json.dump(objects, f, indent=2, ensure_ascii=False)

In [ ]:
for obj in objects:
    if obj['detected_region'] == 4:
        print(f"Object ID: {obj['object_id']}, Detected Region: {obj['detected_region']}, Category: {obj['object_category']}")
        print(f"Centroid: {obj['centroid']}")

Object ID: 20, Detected Region: 4, Category: slippers
Centroid: [5.497086941532025, 3.203002622078035, 0.78393988455881]
Object ID: 23, Detected Region: 4, Category: shelf
Centroid: [6.259544831119763, 3.6164023003927093, 0.6538618327433224]
Object ID: 26, Detected Region: 4, Category: pillow
Centroid: [6.423454235390013, 2.8977171882603407, 0.9400399037724313]
Object ID: 41, Detected Region: 4, Category: bottle
Centroid: [3.6299364377918413, 1.7928165679220818, 2.5505394440267772]
Object ID: 60, Detected Region: 4, Category: Toilet
Centroid: [5.699040832992428, 3.070175887730496, 0.3168102924981393]
Object ID: 66, Detected Region: 4, Category: shoe
Centroid: [5.726194357613794, 3.1664225667822663, 0.42416307190265035]
Object ID: 69, Detected Region: 4, Category: chair
Centroid: [5.38253965228796, 3.140678984671831, 0.4167930387891829]
Object ID: 78, Detected Region: 4, Category: G&H
Centroid: [4.650719637939456, 2.9461628731322014, 0.5880074510551234]
Object ID: 80, Detected Region: 4